# MIA attacks on Length-of-Stay predictor, XGBoost

This notebook trains a **gradient-boosted tree** length-of-stay classifier and audits it with
LeakPro. It is the counterpart to `mimic_LR_main.ipynb` and `mimic_GRUD_main.ipynb`, which
target neural networks.

## Why a tree model

Tabular practice leans heavily on gradient boosting, but every other MIA example in LeakPro
targets a neural network. A tree target is both more representative of how tabular models are
actually built and a harder test of the framework.

## How it works: the `nn.Module` shim

LeakPro's MIA machinery assumes every target is a `torch.nn.Module`. Shadow models are persisted
via `state_dict()`, hashed by iterating state-dict tensors, queried through `PytorchModel`
(`.to(device)`, `.eval()`, `model(tensor)`), and handed to a torch optimizer built from
`model.parameters()`. An XGBoost booster satisfies none of that.

`XGBLOS` in `xgb_target_model.py` is a **compatibility shim**: a real `nn.Module` as far as
LeakPro can tell, but every call is forwarded to a booster stored in a byte buffer. The
serialised booster lives in a `uint8` buffer, which is what makes `state_dict()`, `torch.save`,
and `hash_model` work unchanged.

**This is a workaround, not first-class support.** LeakPro cannot currently audit non-PyTorch
models directly. If you adapt this pattern to scikit-learn or LightGBM, be aware you are
copying a shim.

## Attack compatibility

Because there are no gradients, only logit- and loss-based attacks work:

| Works | Does **not** work |
|---|---|
| `lira`, `rmia`, `base`, `multi_signal_lira`, `yoqo`, `ramia` | `hsj` (backprops through the model), `loss_traj` and `seqmia` (need distillation), `dts` (time series only) |

The incompatible attacks will construct successfully from a config and then fail or return
meaningless scores. `audit_xgb.yaml` documents this too.

## Setup

Requires `xgboost`, which is not part of LeakPro's `mia` extra:

```bash
pip install xgboost
```

Prepare the data first with `mimic_dataset_prep.ipynb`, with `training_method` set to `LR` or
`XGB` in `train_config.yaml` (both produce the flattened `LR_data/` layout that XGBoost needs).


In [ ]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), "../../../"))  # adjust as needed
if project_root not in sys.path:
    sys.path.insert(0, project_root)  # insert at the front to prioritize it


## Train the classifier
### Load the dataset

The dataset is generated by `mimic_dataset_prep.ipynb`. XGBoost consumes the **flattened**
feature matrix, so the data must come from `LR_data/` — set `training_method` to `LR` or `XGB`
in `train_config.yaml` before running the preparation notebook.


In [ ]:
import os
import pickle
import yaml

# Load the config.yaml file
with open("train_config.yaml", "r") as file:
    train_config = yaml.safe_load(file)

# XGB reuses the flattened LR_data layout, so either training_method is valid here.
training_method = train_config["train"]["training_method"]
assert training_method in ("LR", "XGB"), (
    f"training_method is '{training_method}'. XGBoost needs the flattened LR_data layout — "
    "set it to 'LR' or 'XGB' and re-run mimic_dataset_prep.ipynb."
)
data_path = train_config["data"]["data_dir"]
path = os.path.join(data_path, "LR_data")

# File paths
dataset_path = os.path.join(path, "dataset.pkl")
indices_path = os.path.join(path, "indices.pkl")

# Load dataset and indices
if os.path.exists(dataset_path) and os.path.exists(indices_path):
    print("Loading dataset...")

    with open(dataset_path, "rb") as f:
        dataset = pickle.load(f)

    with open(indices_path, "rb") as f:
        indices_dict = pickle.load(f)
        train_indices = indices_dict["train_indices"]
        test_indices = indices_dict["test_indices"]
        early_stop_indices = indices_dict["early_stop_indices"]

    print(f"Loaded dataset and indices from {path}")
    print(f"Data shape: {tuple(dataset.data.shape)}, positive rate: {float(dataset.targets.float().mean()):.4f}")
else:
    print("Dataset not found.\n→ Run 'mimic_dataset_prep.ipynb' to generate the required dataset.\n")


Create data loaders.

XGBoost trains on the full matrix at once rather than in minibatches, but LeakPro drives
training through a `DataLoader`, so the handler drains it into a single array. The batch size
therefore affects only memory traffic, not the fitted model.


In [ ]:
from torch.utils.data import DataLoader
from mimic_data_handler import MIMICUserDataset

data = dataset.data
targets = dataset.targets

train_subset = MIMICUserDataset(data[train_indices], targets[train_indices])
test_subset = MIMICUserDataset(data[test_indices], targets[test_indices])
early_stop_subset = MIMICUserDataset(data[early_stop_indices], targets[early_stop_indices])

# Create DataLoaders
batch_size = train_config["data"]["batch_size"]
train_loader = DataLoader(train_subset, batch_size=batch_size)
test_loader = DataLoader(test_subset, batch_size=batch_size)
early_stop_loader = DataLoader(early_stop_subset, batch_size=batch_size)

print(f"train={len(train_subset)}, test={len(test_subset)}, early_stop={len(early_stop_subset)}")


### Time a single fit

The design matrix is ~7500 features wide. Fit one small booster first to get a feel for the cost
before training the target and then 8 more shadow models on top of it.


In [ ]:
import time
import numpy as np
from xgb_target_model import XGBLOS

n_features = dataset.data.shape[1]
print(f"Number of features: {n_features}")

_probe_x, _probe_y = train_subset.data.numpy(), train_subset.targets.numpy().reshape(-1).astype(np.int64)
_probe = XGBLOS(input_dim=n_features, n_estimators=20, max_depth=6,
                colsample_bytree=train_config["train"]["XGB"]["colsample_bytree"])

_t0 = time.time()
_probe.fit(_probe_x, _probe_y)
_per_round = (time.time() - _t0) / 20
print(f"~{_per_round:.2f}s per boosting round")
print(f"Estimated target fit: {_per_round * train_config['train']['XGB']['n_estimators']:.0f}s")
print(f"Estimated 8 shadow models (~half the data each): {_per_round * train_config['train']['XGB']['n_estimators'] * 8 / 2:.0f}s")


### Train the target model

Hyperparameters come from the `train.XGB` block in `train_config.yaml`. `max_depth` and
`n_estimators` drive the train/test gap, which is what a membership inference attack exploits —
a model that does not overfit leaks little.


In [ ]:
from torch import save
from mimic_model_handler import XGBHandler

# Create model from the config hyperparameters
xgb_params = train_config["train"]["XGB"]
model = XGBLOS(input_dim=n_features, **xgb_params)

# Trees have no optimizer or criterion of their own. The handler builds a stepless placeholder
# optimizer (LeakPro requires one) and reports metrics under BCEWithLogitsLoss so the numbers are
# comparable to the LR and GRU-D targets.
handler = XGBHandler()
criterion = handler.get_criterion()
optimizer = handler.get_optimizer(model)

# Train the model
train_results = XGBHandler().train(train_loader, model)

# Evaluate the model
test_results = XGBHandler().eval(test_loader, model, criterion)

# Store model and its metadata
model = train_results.model
target_dir = "target_XGB"
os.makedirs(target_dir, exist_ok=True)
with open(target_dir + "/target_model.pkl", "wb") as f:
    save(model.state_dict(), f)

# Create metadata to be used by LeakPro
from leakpro import LeakPro
meta_data = LeakPro.make_mia_metadata(train_result = train_results,
                                    optimizer = optimizer,
                                    loss_fn = criterion,
                                    dataloader = train_loader,
                                    test_result = test_results,
                                    epochs = 1,  # trees have no epochs; recorded as 1 and ignored
                                    train_indices = train_indices,
                                    test_indices = test_indices,
                                    dataset_name = train_config["data"]["dataset"])

with open(target_dir + "/model_metadata.pkl", "wb") as f:
    pickle.dump(meta_data, f)

print(f"Saved target model and metadata to {target_dir}/")


### Sanity-check the shim before auditing

If any of these fail, every shadow model produced downstream would be silently wrong. Checking
here costs seconds; discovering it mid-audit costs the whole shadow-model sweep.

1. `init_params` must be non-empty — LeakPro rebuilds shadow models from it, and an empty dict
   would silently give every shadow model default hyperparameters.
2. A `state_dict` round-trip must reproduce identical margins.
3. `hash_model` must agree across the round-trip, since it keys the shadow-model cache.


In [ ]:
from torch import from_numpy
from leakpro.utils.conversion import get_model_init_params
from leakpro.utils.save_load import hash_model

init_params = get_model_init_params(model)
assert init_params, "init_params is empty — shadow models would be built with default hyperparameters"
print(f"init_params: {init_params}")

# Round-trip the model through a state dict, exactly as LeakPro does when loading shadow models.
reloaded = XGBLOS(**init_params)
reloaded.load_state_dict(model.state_dict())

probe = from_numpy(test_subset.data[:256].numpy())
original_margins = model(probe).detach().numpy()
reloaded_margins = reloaded(probe).detach().numpy()

assert np.allclose(original_margins, reloaded_margins, atol=1e-5), "state_dict round-trip changed predictions"
assert hash_model(model) == hash_model(reloaded), "state_dict round-trip changed the model hash"
print(f"Round-trip OK. Model hash: {hash_model(model)[:16]}...")


### The generalisation gap

There are no epochs to plot, so the useful picture is the train/test gap directly. This gap is
the quantity membership inference exploits: the wider it is, the more the model's confidence
distinguishes members from non-members.


In [ ]:
import matplotlib.pyplot as plt

train_acc, test_acc = train_results.metrics.accuracy, test_results.accuracy
train_loss, test_loss = train_results.metrics.loss, test_results.loss
acc_gap = train_acc - test_acc

print(f"Train accuracy: {train_acc:.4f}   Test accuracy: {test_acc:.4f}   Gap: {acc_gap:.4f}")
print(f"Train loss:     {train_loss:.4f}   Test loss:     {test_loss:.4f}")
if acc_gap < 0.02:
    print("\nWARNING: gap < 2pp. The model barely overfits, so expect a weak attack signal.")
    print("Raise max_depth / n_estimators in train_config.yaml for a more informative audit.")

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, (name, values) in zip(axes, [("Accuracy", (train_acc, test_acc)), ("Loss", (train_loss, test_loss))]):
    ax.bar(["Train", "Test"], values, color=["#4C72B0", "#C44E52"])
    ax.set_title(f"{name} (gap = {values[0] - values[1]:+.4f})")
    ax.set_ylabel(name)
plt.tight_layout()
plt.show()


## Attack the XGBoost model

The audit is configured by `audit_xgb.yaml`, a standalone file rather than another commented-out
block in `audit.yaml`:

```yaml
target:
  module_path: "./xgb_target_model.py"
  model_class: "XGBLOS"
  target_folder: "./target_XGB"
  data_path: "./data/LR_data/dataset.pkl"
```

It runs online LiRA (8 shadow models) and offline RMIA (4 shadow models). The `population`
attack is deliberately excluded: it reports an inverted ROC for every target (see the note in
`audit_xgb.yaml`). Shadow models are cached, so re-running this cell reuses them — the log will say
`Reusing 8 cached shadow model(s)`. Changing any XGBoost hyperparameter invalidates that cache,
because the shadow-model training signature includes `init_params`.


In [ ]:
from leakpro import LeakPro
from mimic_model_handler import XGBHandler as InputHandler

# Read the config file
config_path = "audit_xgb.yaml"

# Instantiate leakpro object
leakpro = LeakPro(InputHandler, config_path)

# Run the audit
mia_results = leakpro.run_audit(create_pdf=True)


In [ ]:
FPR_KEYS = ["TPR@0%FPR", "TPR@0.01%FPR", "TPR@0.1%FPR", "TPR@1%FPR", "TPR@10%FPR"]
COL_W = max(len(k) for k in FPR_KEYS)


def format_mia_result(result) -> str:
    lines = [result.result_name, "  Config:"]
    lines += [f"    {key}: {value}" for key, value in result.result_config.items()]

    table = result.fixed_fpr_table
    lines.append("  TPR @ fixed FPR")
    lines.append("    " + " | ".join(f"{key:>{COL_W}}" for key in FPR_KEYS))
    lines.append("    " + " | ".join(
        f"{table[key]:>{COL_W}.4f}" if key in table else f"{'n/a':>{COL_W}}"
        for key in FPR_KEYS
    ))
    return "\n".join(lines)


for result in mia_results:
    print(format_mia_result(result), end="\n\n")

In [ ]:
from IPython.display import Image

Image("leakpro_output_xgb/results/ROC.png", width=600)

## Reading the results

The report lands in `leakpro_output_xgb/`. Read **TPR at low FPR**, not AUC: average-case metrics
hide the worst-case leakage that matters for individual records.

Two sanity conditions on the LiRA result:

- **AUC > 0.5.** A sub-0.5 AUC means the signal orientation is inverted. `MIAResult` does not
  flip it for you.
- **TPR@1%FPR above the 1% diagonal.** At or below it, the attack is no better than guessing in
  the region of interest, regardless of what the AUC says.

### Caveats

- **The shim.** `XGBLOS` is a compatibility layer, not first-class support for non-PyTorch
  models. Gradient- and distillation-based attacks are unavailable, and the recorded optimizer,
  criterion and epoch count in the model metadata are placeholders.
- **Determinism.** With `subsample` or `colsample_bytree` below 1.0, XGBoost is seed-dependent.
  `random_state` is pinned in `train_config.yaml`, but shadow models are not bit-reproducible
  across runs while the seeding overhaul (issue #325) is open.
- **No DP baseline.** `mimic_GRUD_DPSGD_main.ipynb` provides a differentially private comparison
  for the neural target. There is no equivalent DP tree model here, so these numbers have no
  privacy-protected counterpart to be read against.
- **A weak result is still a result.** If the attack lands near chance, report that rather than
  tuning hyperparameters until the numbers look impressive — the honest finding is that this
  particular model configuration leaks little.
